# 01 — Executive Brief One Page

## Propósito

Producir una página ejecutiva que diga qué muestran los números disponibles, sin intentar probar toda la historia ni cerrar todas las discusiones metodológicas.

## Pregunta humana

**¿Qué tiene que entender alguien en 3 minutos sobre renta, costos, margen, retiros, funding y deuda?**

## Usuarios

- Hermanos.
- Padre, en versión más simple.
- Abogado, como portada económica.
- Matías, para no sobreexplicar.

## Contrato del notebook

Este notebook:

- consume outputs canónicos de `out/`;
- prefiere métricas limpias cuando existen;
- usa métricas legacy solo con caveat;
- no redefine la contabilidad base;
- no intenta cerrar el saldo jurídico final por actor;
- produce un brief de una página, KPIs, caveats y fragmentos HTML/Markdown.

## Criterio de cierre

El notebook se considera terminado si exporta:

- 5–7 KPIs;
- 5 hallazgos;
- al menos 3 caveats;
- referencias a tablas de soporte;
- warnings cuando use métricas legacy o falten métricas limpias.


## 1. Load latest run context

In [1]:
from pathlib import Path
import os
import json
import re
from datetime import datetime
from typing import Optional, Iterable, Any

import pandas as pd
from IPython.display import display, Markdown, HTML

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

def find_repo_root(start: Optional[Path] = None) -> Path:
    """Find the repo root by walking upward until accounting/ and Makefile are visible."""
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "accounting").is_dir() and (p / "Makefile").exists():
            return p
    # Fallback for notebooks/accounting execution.
    for p in candidates:
        if (p / "out").is_dir() and (p / "accounting").is_dir():
            return p
    return start

REPO = find_repo_root()
OUT = REPO / "out"

METRICS = OUT / "metrics" / "latest"
DEBT = OUT / "debt_resolution" / "latest"
RUN = OUT / "run" / "accounting" / "latest"
HUMAN = OUT / "human_reports" / "latest"

# Period/output label can be overridden from env.
PERIOD_LABEL = os.environ.get("ACCOUNTING_BRIEF_PERIOD") or os.environ.get("ACCOUNTING_PERIOD") or "latest"
PACK = OUT / "professional_pack" / PERIOD_LABEL / "executive_brief"
PACK.mkdir(parents=True, exist_ok=True)

print(f"REPO    = {REPO}")
print(f"METRICS = {METRICS}")
print(f"DEBT    = {DEBT}")
print(f"RUN     = {RUN}")
print(f"HUMAN   = {HUMAN}")
print(f"PACK    = {PACK}")


REPO    = /home/matias/repos/accounting-backend
METRICS = /home/matias/repos/accounting-backend/out/metrics/latest
DEBT    = /home/matias/repos/accounting-backend/out/debt_resolution/latest
RUN     = /home/matias/repos/accounting-backend/out/run/accounting/latest
HUMAN   = /home/matias/repos/accounting-backend/out/human_reports/latest
PACK    = /home/matias/repos/accounting-backend/out/professional_pack/latest/executive_brief


## 2. Load canonical frontier contract and series


In [2]:
def read_csv_safe(path: Path, **kwargs) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return pd.DataFrame()
    try:
        df = pd.read_csv(path, **kwargs)
        print(f"[loaded] {path} — {df.shape[0]:,} rows × {df.shape[1]:,} cols")
        return df
    except Exception as e:
        print(f"[error] could not read {path}: {e}")
        return pd.DataFrame()

def read_json_safe(path: Path) -> dict:
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception as e:
        print(f"[error] could not read {path}: {e}")
        return {}

metric_contract_frontier = read_csv_safe(METRICS / "metric_contract_frontier.csv")
frontend_metric_series = read_csv_safe(METRICS / "frontend_metric_series.csv")
metrics_frontier_qa = read_csv_safe(METRICS / "metrics_frontier_qa.csv")

# Legacy files are loaded only for compatibility/caveat display, not for semantic decisions.
metric_registry_legacy = read_csv_safe(METRICS / "metric_registry.csv")
metric_values_legacy = read_csv_safe(METRICS / "metric_values.csv")
validation_report = read_csv_safe(METRICS / "validation_report.csv")

# Downstream cells keep their historical variable names, but they now point to the canonical frontier.
metric_registry = metric_contract_frontier.copy()
metric_values = frontend_metric_series.copy()

story_manifest = read_json_safe(HUMAN / "balance_human_v2" / "story_manifest.json")
build_manifest = read_json_safe(METRICS / "build_manifest.json")

print("Canonical notebook contract: use frontend_metric_series.csv for values and metric_contract_frontier.csv for labels/caveats.")
print("No OPEX/cash semantics are inferred in this notebook.")


[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/metric_contract_frontier.csv — 28 rows × 18 cols
[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/frontend_metric_series.csv — 1,283 rows × 17 cols
[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/metrics_frontier_qa.csv — 20 rows × 4 cols
[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/metric_registry.csv — 22 rows × 19 cols
[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/metric_values.csv — 418 rows × 10 cols
[loaded] /home/matias/repos/accounting-backend/out/metrics/latest/validation_report.csv — 2 rows × 4 cols
Canonical notebook contract: use frontend_metric_series.csv for values and metric_contract_frontier.csv for labels/caveats.
No OPEX/cash semantics are inferred in this notebook.


## 3. Resolve clean vs legacy metrics

In [3]:
VALUE_COL_CANDIDATES = [
    "value", "metric_value", "amount", "amount_ars", "value_ars",
    "current_value", "current_value_ARS", "total", "total_ARS"
]

PERIOD_COL_CANDIDATES = [
    "period", "period_key", "period_label", "date", "month",
    "year", "period_start", "period_end", "as_of_date"
]

FREQ_COL_CANDIDATES = [
    "freq", "frequency", "period_freq", "period_type", "granularity", "time_grain"
]

def first_existing_col(df: pd.DataFrame, candidates: Iterable[str]) -> Optional[str]:
    if df is None or df.empty:
        return None
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in df.columns:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None

def infer_value_col(df: pd.DataFrame) -> Optional[str]:
    c = first_existing_col(df, VALUE_COL_CANDIDATES)
    if c:
        return c
    numeric_cols = list(df.select_dtypes(include="number").columns)
    if numeric_cols:
        # Avoid obvious metadata columns when possible.
        for col in numeric_cols:
            if col.lower() not in {"year", "month", "sort_key"}:
                return col
        return numeric_cols[0]
    return None

def infer_period_col(df: pd.DataFrame) -> Optional[str]:
    return first_existing_col(df, PERIOD_COL_CANDIDATES)

def infer_freq_col(df: pd.DataFrame) -> Optional[str]:
    return first_existing_col(df, FREQ_COL_CANDIDATES)

def normalize_metric_values(df: pd.DataFrame) -> pd.DataFrame:
    """Return a normalized long-ish metric table with helper columns if possible."""
    if df.empty:
        return df.copy()
    out = df.copy()
    value_col = infer_value_col(out)
    period_col = infer_period_col(out)
    freq_col = infer_freq_col(out)

    if value_col:
        out["_value"] = pd.to_numeric(out[value_col], errors="coerce")
    else:
        out["_value"] = pd.NA

    if period_col:
        out["_period_raw"] = out[period_col].astype(str)
        out["_period_dt"] = pd.to_datetime(out[period_col], errors="coerce")
    else:
        out["_period_raw"] = ""
        out["_period_dt"] = pd.NaT

    if freq_col:
        out["_freq"] = out[freq_col].astype(str).str.upper()
    else:
        out["_freq"] = ""

    # Prefer monthly if available, then quarterly, then yearly, then any.
    return out

mv = normalize_metric_values(metric_values)

def registry_row(metric_id: str) -> dict:
    if metric_registry.empty or "metric_id" not in metric_registry.columns:
        return {}
    rows = metric_registry.loc[metric_registry["metric_id"].astype(str) == metric_id]
    if rows.empty:
        return {}
    return rows.iloc[0].to_dict()

def metric_type(metric_id: str) -> str:
    row = registry_row(metric_id)
    t = str(row.get("flow_or_stock", "") or row.get("metric_type", "") or "").lower()
    if t:
        return t
    # Fallback from naming.
    if metric_id.startswith("BS.") or ".CLOSING" in metric_id or ".OPEN" in metric_id:
        return "stock"
    return "flow"

def metric_label(metric_id: str) -> str:
    row = registry_row(metric_id)
    return str(row.get("label") or metric_id)

def metric_exists(metric_id: str) -> bool:
    return (not mv.empty) and ("metric_id" in mv.columns) and (mv["metric_id"].astype(str) == metric_id).any()

def choose_metric(preferred: list[str], legacy: list[str] | None = None) -> tuple[Optional[str], bool, str]:
    """Return chosen metric_id, used_legacy flag, reason."""
    legacy = legacy or []
    for m in preferred:
        if metric_exists(m):
            return m, False, "preferred"
    for m in legacy:
        if metric_exists(m):
            return m, True, "legacy_fallback"
    return None, False, "missing"

def metric_summary(metric_id: str, last_n: int = 6) -> dict:
    """Compute a robust summary for a metric across the latest available periods."""
    if not metric_id or mv.empty or "metric_id" not in mv.columns:
        return {}
    rows = mv.loc[mv["metric_id"].astype(str) == metric_id].copy()
    if rows.empty:
        return {}

    # Prefer monthly rows if a freq column is present and M exists.
    if "_freq" in rows.columns and rows["_freq"].notna().any():
        freq_values = set(rows["_freq"].dropna().astype(str).str.upper())
        for preferred_freq in ["M", "MONTH", "MONTHLY"]:
            monthly = rows.loc[rows["_freq"].astype(str).str.upper() == preferred_freq]
            if not monthly.empty:
                rows = monthly.copy()
                break

    # Sort by parsed period if possible, else raw period, else original order.
    if "_period_dt" in rows.columns and rows["_period_dt"].notna().any():
        rows = rows.sort_values("_period_dt")
    elif "_period_raw" in rows.columns:
        rows = rows.sort_values("_period_raw")
    else:
        rows = rows.reset_index(drop=True)

    value_series = pd.to_numeric(rows["_value"], errors="coerce").dropna()
    if value_series.empty:
        return {"metric_id": metric_id, "status": "no_numeric_value"}

    latest_row = rows.dropna(subset=["_value"]).iloc[-1]
    latest_value = float(latest_row["_value"])
    latest_period = str(latest_row.get("_period_raw", ""))

    recent = rows.dropna(subset=["_value"]).tail(last_n)
    recent_values = pd.to_numeric(recent["_value"], errors="coerce").dropna()

    mtype = metric_type(metric_id)
    if mtype == "stock":
        main_value = latest_value
        main_label = "Saldo/cierre"
        total_6 = None
        avg_m = None
    else:
        main_value = float(recent_values.sum())
        main_label = f"Total últimos {len(recent_values)} períodos"
        total_6 = main_value
        avg_m = float(recent_values.mean()) if len(recent_values) else None

    return {
        "metric_id": metric_id,
        "label": metric_label(metric_id),
        "metric_type": mtype,
        "periods_used": int(len(recent_values)),
        "latest_period": latest_period,
        "latest_value": latest_value,
        "main_label": main_label,
        "main_value": main_value,
        "total_last_n": total_6,
        "avg_period": avg_m,
        "source": "frontend_metric_series.csv",
        "status": "ok",
    }

def money_ars(x: Any) -> str:
    try:
        if pd.isna(x):
            return "s/d"
        x = float(x)
    except Exception:
        return "s/d"
    sign = "-" if x < 0 else ""
    x = abs(x)
    return f"{sign}$ {x:,.0f}".replace(",", ".")

def short_number(x: Any) -> str:
    try:
        if pd.isna(x):
            return "s/d"
        x = float(x)
    except Exception:
        return "s/d"
    return f"{x:,.0f}".replace(",", ".")

KPI_SPECS = [
    {
        "block": "Renta",
        "question": "¿Cuánta renta entró?",
        "preferred": ["IS.RENT.TOTAL"],
        "legacy": [],
        "fallback_note": "Si falta, revisar metric_views de renta/rent.",
        "support": "metric_values.csv; metric_views/rent*.csv o renta*.csv",
    },
    {
        "block": "Costos operativos",
        "question": "¿Cuánto costó sostener la operación?",
        "preferred": ["IS.OPEX.PROPERTY"],
        "legacy": [],
        "fallback_note": "Si falta, revisar v_opex_category_monthly.csv / opex rollups.",
        "support": "frontend_metric_series.csv; monthly_operating_statement.csv",
    },
    {
        "block": "Resultado operativo",
        "question": "¿Qué quedó después de costos, sin contar funding familiar?",
        "preferred": ["IS.NET.OPERATING"],
        "legacy": ["IS.NET.AFTER_COSTS"],
        "fallback_note": "Legacy puede mezclar contribuciones si depende de IS.INCOME.TOTAL.",
        "support": "frontend_metric_series.csv; metric_contract_frontier.csv",
    },
    {
        "block": "Funding familiar",
        "question": "¿Cuánto funding/aporte familiar sostuvo la caja?",
        "preferred": ["FUND.CONTRIB.TOTAL"],
        "legacy": ["IS.CONTRIB.TOTAL"],
        "fallback_note": "Legacy vive bajo IS pero semánticamente es funding.",
        "support": "frontend_metric_series.csv; monthly_operating_statement.csv",
    },
    {
        "block": "Retiros / distribución",
        "question": "¿Qué salidas distributivas o retiros hubo?",
        "preferred": ["DIST.DRAWS.PERSONAL"],
        "legacy": ["IS.DRAWS.PERSONAL"],
        "fallback_note": "Legacy vive bajo IS pero semánticamente es distribución/draw.",
        "support": "metric_values.csv; metric_views/*draws*.csv",
    },
    {
        "block": "Caja cierre",
        "question": "¿Con qué caja visible cerró el período?",
        "preferred": ["BS.CASH.TOTAL"],
        "legacy": [],
        "fallback_note": "Si falta, revisar balance_cash_q/y o daily_cash_position.",
        "support": "metric_values.csv; balance_cash_*.csv; daily_cash_position.csv",
    },
    {
        "block": "Deuda interna",
        "question": "¿Qué deuda/exposición interna aparece abierta?",
        "preferred": ["ID.TOTAL.CLOSING_BALANCE", "ID.MATIAS.CLOSING_BALANCE"],
        "legacy": ["BS.DEBT.TOTAL.OPEN"],
        "fallback_note": "Si falta ID.*, usar debt_balance_monthly/yearly o BS.DEBT.TOTAL.OPEN.",
        "support": "metric_values.csv; debt_balance_*.csv; debt_open_items.csv",
    },
]



## 4. Build KPI cards

In [4]:
kpi_rows = []
caveat_rows = []

for spec in KPI_SPECS:
    chosen, used_legacy, reason = choose_metric(spec["preferred"], spec.get("legacy", []))
    if chosen:
        summary = metric_summary(chosen, last_n=6)
        status = summary.get("status", "missing")
    else:
        summary = {}
        status = "missing"

    row = {
        "block": spec["block"],
        "question": spec["question"],
        "preferred_metric": ", ".join(spec["preferred"]),
        "legacy_metric": ", ".join(spec.get("legacy", [])),
        "chosen_metric_id": chosen or "",
        "used_legacy": used_legacy,
        "status": status,
        "metric_type": summary.get("metric_type", ""),
        "periods_used": summary.get("periods_used", ""),
        "latest_period": summary.get("latest_period", ""),
        "main_label": summary.get("main_label", ""),
        "main_value": summary.get("main_value", pd.NA),
        "main_value_fmt": money_ars(summary.get("main_value", pd.NA)),
        "latest_value": summary.get("latest_value", pd.NA),
        "latest_value_fmt": money_ars(summary.get("latest_value", pd.NA)),
        "avg_period": summary.get("avg_period", pd.NA),
        "avg_period_fmt": money_ars(summary.get("avg_period", pd.NA)),
        "support": spec["support"],
        "fallback_note": spec["fallback_note"],
    }
    kpi_rows.append(row)

    if status != "ok":
        caveat_rows.append({
            "severity": "warning",
            "topic": spec["block"],
            "message": f"No se encontró una métrica usable para {spec['block']}. Revisar fallback: {spec['fallback_note']}",
            "metric_id": chosen or "",
        })
    elif used_legacy:
        caveat_rows.append({
            "severity": "warning",
            "topic": spec["block"],
            "message": f"Se usó métrica legacy `{chosen}`. {spec['fallback_note']}",
            "metric_id": chosen,
        })

executive_kpis = pd.DataFrame(kpi_rows)

# Minimum generic caveats that should always travel with the one-page brief.
base_caveats = [
    {
        "severity": "scope",
        "topic": "Alcance",
        "message": "Este brief resume patrones económicos principales; no intenta cerrar toda la historia ni reemplazar auditoría externa.",
        "metric_id": "",
    },
    {
        "severity": "scope",
        "topic": "Prueba",
        "message": "Los números sostienen patrones de renta, costos, caja, funding y deuda; no prueban por sí solos responsabilidad jurídica individual definitiva.",
        "metric_id": "",
    },
    {
        "severity": "method",
        "topic": "Neteo por actor",
        "message": "El saldo final fino por actor requiere notebook específico de deuda/neteo y soporte documental granular.",
        "metric_id": "",
    },
]
executive_caveats = pd.DataFrame(base_caveats + caveat_rows)

display(executive_kpis)
display(executive_caveats)


,block,question,preferred_metric,legacy_metric,chosen_metric_id,used_legacy,status,metric_type,periods_used,latest_period,main_label,main_value,main_value_fmt,latest_value,latest_value_fmt,avg_period,avg_period_fmt,support,fallback_note
0,Renta,¿Cuánta renta entró?,IS.RENT.TOTAL,,IS.RENT.TOTAL,False,ok,flow,6,2026-06,Total últimos 6 períodos,13474140.0,$ 13.474.140,380.0,$ 380,2245690.0,$ 2.245.690,metric_values.csv; metric_views/rent*.csv o re...,"Si falta, revisar metric_views de renta/rent."
1,Costos operativos,¿Cuánto costó sostener la operación?,IS.OPEX.PROPERTY,,IS.OPEX.PROPERTY,False,ok,flow,6,2026-07,Total últimos 6 períodos,1362375.46,$ 1.362.375,182961.23,$ 182.961,227062.576667,$ 227.063,frontend_metric_series.csv; monthly_operating_...,"Si falta, revisar v_opex_category_monthly.csv ..."
2,Resultado operativo,"¿Qué quedó después de costos, sin contar fundi...",IS.NET.OPERATING,IS.NET.AFTER_COSTS,IS.NET.OPERATING,False,ok,flow,6,2026-07,Total últimos 6 períodos,7631764.54,$ 7.631.765,-182961.23,-$ 182.961,1271960.756667,$ 1.271.961,frontend_metric_series.csv; metric_contract_fr...,Legacy puede mezclar contribuciones si depende...
3,Funding familiar,¿Cuánto funding/aporte familiar sostuvo la caja?,FUND.CONTRIB.TOTAL,IS.CONTRIB.TOTAL,FUND.CONTRIB.TOTAL,False,ok,flow,6,2026-07,Total últimos 6 períodos,110000.0,$ 110.000,0.0,$ 0,18333.333333,$ 18.333,frontend_metric_series.csv; monthly_operating_...,Legacy vive bajo IS pero semánticamente es fun...
4,Retiros / distribución,¿Qué salidas distributivas o retiros hubo?,DIST.DRAWS.PERSONAL,IS.DRAWS.PERSONAL,DIST.DRAWS.PERSONAL,False,ok,flow,6,2026-07,Total últimos 6 períodos,8723764.0,$ 8.723.764,0.0,$ 0,1453960.666667,$ 1.453.961,metric_values.csv; metric_views/*draws*.csv,Legacy vive bajo IS pero semánticamente es dis...
5,Caja cierre,¿Con qué caja visible cerró el período?,BS.CASH.TOTAL,,,False,missing,,,,,<NA>,s/d,<NA>,s/d,<NA>,s/d,metric_values.csv; balance_cash_*.csv; daily_c...,"Si falta, revisar balance_cash_q/y o daily_cas..."
6,Deuda interna,¿Qué deuda/exposición interna aparece abierta?,"ID.TOTAL.CLOSING_BALANCE, ID.MATIAS.CLOSING_BA...",BS.DEBT.TOTAL.OPEN,,False,missing,,,,,<NA>,s/d,<NA>,s/d,<NA>,s/d,metric_values.csv; debt_balance_*.csv; debt_op...,"Si falta ID.*, usar debt_balance_monthly/yearl..."


,severity,topic,message,metric_id
0,scope,Alcance,Este brief resume patrones económicos principa...,
1,scope,Prueba,"Los números sostienen patrones de renta, costo...",
2,method,Neteo por actor,El saldo final fino por actor requiere noteboo...,
3,warning,Caja cierre,No se encontró una métrica usable para Caja ci...,
4,warning,Deuda interna,No se encontró una métrica usable para Deuda i...,


## 5. Build one-page narrative

In [5]:
def get_kpi(block: str) -> dict:
    rows = executive_kpis.loc[executive_kpis["block"] == block]
    if rows.empty:
        return {}
    return rows.iloc[0].to_dict()

def kpi_sentence(block: str, label: Optional[str] = None) -> str:
    k = get_kpi(block)
    if not k:
        return f"- **{label or block}:** s/d."
    chosen = k.get("chosen_metric_id") or "sin métrica"
    value = k.get("main_value_fmt") or "s/d"
    avg = k.get("avg_period_fmt") or "s/d"
    latest = k.get("latest_value_fmt") or "s/d"
    mtype = k.get("metric_type", "")
    legacy_note = " _(legacy)_" if bool(k.get("used_legacy")) else ""
    if mtype == "stock":
        return f"- **{label or block}:** {latest} al cierre observado (`{chosen}`{legacy_note})."
    return f"- **{label or block}:** {value}; promedio por período {avg} (`{chosen}`{legacy_note})."

def warning_block() -> str:
    warnings = executive_caveats.loc[executive_caveats["severity"].isin(["warning", "method", "scope"])]
    lines = []
    for _, r in warnings.head(6).iterrows():
        lines.append(f"- **{r['topic']}:** {r['message']}")
    return "\n".join(lines)

legacy_used = executive_kpis.loc[executive_kpis["used_legacy"] == True, "chosen_metric_id"].dropna().astype(str).tolist()
missing_blocks = executive_kpis.loc[executive_kpis["status"] != "ok", "block"].dropna().astype(str).tolist()

generated_at = datetime.now().strftime("%Y-%m-%d %H:%M")

hallazgos = [
    "Hubo renta relevante y, por lo tanto, la discusión no se explica solamente por ausencia absoluta de ingresos.",
    "Hubo costos operativos reales: sostener el patrimonio requirió impuestos, servicios, mantenimiento u otros egresos identificables.",
    "El resultado operativo debe leerse separado del funding familiar; los aportes no deben confundirse con ingresos económicos de la operación.",
    "Los retiros o salidas distributivas deben compararse contra el margen operativo y la caja disponible, no contra la renta bruta.",
    "La pregunta central de gobernanza es si hubo trazabilidad, rendición y prioridad suficiente de costos/conservación antes de distribuir.",
]

brief_md = f"""# Resumen ejecutivo — qué muestran los números

_Generado: {generated_at}_  
_Período/salida: `{PERIOD_LABEL}`_

## Lectura en 3 minutos

Este brief resume el patrón económico principal con los datos disponibles. No intenta cerrar toda la historia, fijar responsabilidades jurídicas definitivas ni reemplazar una auditoría externa. Su función es ordenar la conversación: renta, costos, resultado operativo, funding familiar, retiros, caja y deuda.

## KPIs principales

{kpi_sentence("Renta")}
{kpi_sentence("Costos operativos")}
{kpi_sentence("Resultado operativo")}
{kpi_sentence("Funding familiar")}
{kpi_sentence("Retiros / distribución")}
{kpi_sentence("Caja cierre")}
{kpi_sentence("Deuda interna")}

## 5 hallazgos defendibles

1. {hallazgos[0]}
2. {hallazgos[1]}
3. {hallazgos[2]}
4. {hallazgos[3]}
5. {hallazgos[4]}

## Caveats y límites

{warning_block()}

## Tablas de soporte

- `metric_registry.csv` — contrato de métricas y metadatos semánticos.
- `metric_values.csv` — valores calculados por métrica y período.
- `validation_report.csv` — controles de calidad disponibles.
- `metric_views/` — rollups de renta, costos, draws, cash y deuda cuando existan.
- `debt_resolution/latest/` — open items, repayment events, timeline y balances de deuda.
- `ledger_canonical.csv` — fuente canónica de auditoría.

## Warnings automáticos

- Métricas legacy usadas: {", ".join(legacy_used) if legacy_used else "ninguna"}.
- Bloques sin métrica usable: {", ".join(missing_blocks) if missing_blocks else "ninguno"}.

## Próximo paso recomendado

Usar este brief como portada. Si alguien cuestiona una afirmación, bajar al cuadro de soporte correspondiente: primero métrica, luego rollup, luego drilldown o ledger.
"""

display(Markdown(brief_md))


# Resumen ejecutivo — qué muestran los números

_Generado: 2026-07-11 15:21_  
_Período/salida: `latest`_

## Lectura en 3 minutos

Este brief resume el patrón económico principal con los datos disponibles. No intenta cerrar toda la historia, fijar responsabilidades jurídicas definitivas ni reemplazar una auditoría externa. Su función es ordenar la conversación: renta, costos, resultado operativo, funding familiar, retiros, caja y deuda.

## KPIs principales

- **Renta:** $ 13.474.140; promedio por período $ 2.245.690 (`IS.RENT.TOTAL`).
- **Costos operativos:** $ 1.362.375; promedio por período $ 227.063 (`IS.OPEX.PROPERTY`).
- **Resultado operativo:** $ 7.631.765; promedio por período $ 1.271.961 (`IS.NET.OPERATING`).
- **Funding familiar:** $ 110.000; promedio por período $ 18.333 (`FUND.CONTRIB.TOTAL`).
- **Retiros / distribución:** $ 8.723.764; promedio por período $ 1.453.961 (`DIST.DRAWS.PERSONAL`).
- **Caja cierre:** s/d; promedio por período s/d (`sin métrica`).
- **Deuda interna:** s/d; promedio por período s/d (`sin métrica`).

## 5 hallazgos defendibles

1. Hubo renta relevante y, por lo tanto, la discusión no se explica solamente por ausencia absoluta de ingresos.
2. Hubo costos operativos reales: sostener el patrimonio requirió impuestos, servicios, mantenimiento u otros egresos identificables.
3. El resultado operativo debe leerse separado del funding familiar; los aportes no deben confundirse con ingresos económicos de la operación.
4. Los retiros o salidas distributivas deben compararse contra el margen operativo y la caja disponible, no contra la renta bruta.
5. La pregunta central de gobernanza es si hubo trazabilidad, rendición y prioridad suficiente de costos/conservación antes de distribuir.

## Caveats y límites

- **Alcance:** Este brief resume patrones económicos principales; no intenta cerrar toda la historia ni reemplazar auditoría externa.
- **Prueba:** Los números sostienen patrones de renta, costos, caja, funding y deuda; no prueban por sí solos responsabilidad jurídica individual definitiva.
- **Neteo por actor:** El saldo final fino por actor requiere notebook específico de deuda/neteo y soporte documental granular.
- **Caja cierre:** No se encontró una métrica usable para Caja cierre. Revisar fallback: Si falta, revisar balance_cash_q/y o daily_cash_position.
- **Deuda interna:** No se encontró una métrica usable para Deuda interna. Revisar fallback: Si falta ID.*, usar debt_balance_monthly/yearly o BS.DEBT.TOTAL.OPEN.

## Tablas de soporte

- `metric_registry.csv` — contrato de métricas y metadatos semánticos.
- `metric_values.csv` — valores calculados por métrica y período.
- `validation_report.csv` — controles de calidad disponibles.
- `metric_views/` — rollups de renta, costos, draws, cash y deuda cuando existan.
- `debt_resolution/latest/` — open items, repayment events, timeline y balances de deuda.
- `ledger_canonical.csv` — fuente canónica de auditoría.

## Warnings automáticos

- Métricas legacy usadas: ninguna.
- Bloques sin métrica usable: Caja cierre, Deuda interna.

## Próximo paso recomendado

Usar este brief como portada. Si alguien cuestiona una afirmación, bajar al cuadro de soporte correspondiente: primero métrica, luego rollup, luego drilldown o ledger.


## 6. Export markdown/html/pdf-ready fragments

In [6]:
def markdown_to_simple_html(md: str, title: str = "Resumen ejecutivo") -> str:
    """Small dependency-free Markdown-ish renderer for this brief."""
    import html
    lines = md.splitlines()
    html_lines = []
    in_ul = False
    for line in lines:
        raw = line
        line = line.rstrip()
        if not line:
            if in_ul:
                html_lines.append("</ul>")
                in_ul = False
            html_lines.append("")
            continue
        if line.startswith("# "):
            if in_ul:
                html_lines.append("</ul>")
                in_ul = False
            html_lines.append(f"<h1>{html.escape(line[2:])}</h1>")
        elif line.startswith("## "):
            if in_ul:
                html_lines.append("</ul>")
                in_ul = False
            html_lines.append(f"<h2>{html.escape(line[3:])}</h2>")
        elif re.match(r"^\d+\.\s+", line):
            if in_ul:
                html_lines.append("</ul>")
                in_ul = False
            txt = re.sub(r"^\d+\.\s+", "", line)
            html_lines.append(f"<p class='numbered'>{html.escape(raw)}</p>")
        elif line.startswith("- "):
            if not in_ul:
                html_lines.append("<ul>")
                in_ul = True
            html_lines.append(f"<li>{html.escape(line[2:])}</li>")
        else:
            if in_ul:
                html_lines.append("</ul>")
                in_ul = False
            html_lines.append(f"<p>{html.escape(line)}</p>")
    if in_ul:
        html_lines.append("</ul>")

    body = "\n".join(html_lines)
    # Minimal replacements for inline code and bold after escaping.
    body = re.sub(r"`([^`]+)`", r"<code>\1</code>", body)
    body = re.sub(r"\*\*([^*]+)\*\*", r"<strong>\1</strong>", body)
    body = body.replace("_Generado:", "<em>Generado:").replace("_Período/salida:", "<em>Período/salida:")
    body = body.replace("`_", "`</em>").replace("._", ".</em>")

    return f"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>{html.escape(title)}</title>
<style>
body {{
  font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
  max-width: 880px;
  margin: 40px auto;
  padding: 0 24px 48px;
  color: #1f2937;
  line-height: 1.45;
}}
h1 {{
  font-size: 30px;
  border-bottom: 2px solid #d1d5db;
  padding-bottom: 10px;
}}
h2 {{
  margin-top: 28px;
  font-size: 21px;
  color: #111827;
}}
code {{
  background: #f3f4f6;
  padding: 2px 5px;
  border-radius: 4px;
}}
ul {{
  padding-left: 22px;
}}
li {{
  margin: 6px 0;
}}
.numbered {{
  margin-left: 8px;
}}
.footer {{
  margin-top: 36px;
  padding-top: 12px;
  border-top: 1px solid #e5e7eb;
  color: #6b7280;
  font-size: 13px;
}}
</style>
</head>
<body>
{body}
<div class="footer">Generated by notebooks/accounting/01_executive_brief_one_page.ipynb</div>
</body>
</html>
"""

brief_html = markdown_to_simple_html(brief_md)

md_path = PACK / "executive_brief_1p.md"
html_path = PACK / "executive_brief_1p.html"
kpis_path = PACK / "executive_kpis.csv"
caveats_path = PACK / "executive_caveats.csv"

md_path.write_text(brief_md, encoding="utf-8")
html_path.write_text(brief_html, encoding="utf-8")
executive_kpis.to_csv(kpis_path, index=False)
executive_caveats.to_csv(caveats_path, index=False)

print("Exported:")
for p in [md_path, html_path, kpis_path, caveats_path]:
    print(f" - {p.relative_to(REPO)}")

display(HTML(f"<p><strong>HTML preview written to:</strong> <code>{html_path}</code></p>"))


Exported:
 - out/professional_pack/latest/executive_brief/executive_brief_1p.md
 - out/professional_pack/latest/executive_brief/executive_brief_1p.html
 - out/professional_pack/latest/executive_brief/executive_kpis.csv
 - out/professional_pack/latest/executive_brief/executive_caveats.csv


## 7. QA: missing metrics and caveats

In [7]:
qa_rows = []

required_outputs = {
    "executive_brief_1p.md": PACK / "executive_brief_1p.md",
    "executive_brief_1p.html": PACK / "executive_brief_1p.html",
    "executive_kpis.csv": PACK / "executive_kpis.csv",
    "executive_caveats.csv": PACK / "executive_caveats.csv",
}

for name, path in required_outputs.items():
    qa_rows.append({
        "check": f"output_exists:{name}",
        "ok": path.exists(),
        "detail": str(path.relative_to(REPO)) if path.exists() else str(path),
    })

qa_rows.append({
    "check": "kpi_count_between_5_and_7",
    "ok": 5 <= len(executive_kpis) <= 7,
    "detail": f"{len(executive_kpis)} KPI rows",
})

qa_rows.append({
    "check": "has_at_least_3_caveats",
    "ok": len(executive_caveats) >= 3,
    "detail": f"{len(executive_caveats)} caveat rows",
})

qa_rows.append({
    "check": "legacy_warning_if_legacy_used",
    "ok": (not legacy_used) or any(executive_caveats["message"].str.contains("legacy", case=False, na=False)),
    "detail": ", ".join(legacy_used) if legacy_used else "no legacy metrics used",
})

qa_rows.append({
    "check": "missing_metrics_are_visible",
    "ok": (not missing_blocks) or any(executive_caveats["message"].str.contains("No se encontró", case=False, na=False)),
    "detail": ", ".join(missing_blocks) if missing_blocks else "no missing blocks",
})

qa_report = pd.DataFrame(qa_rows)
qa_path = PACK / "executive_brief_qa.csv"
qa_report.to_csv(qa_path, index=False)

display(qa_report)

if not qa_report["ok"].all():
    display(Markdown("### QA warnings\nHay checks que no pasaron. Revisar `executive_brief_qa.csv` y los caveats antes de circular el brief."))
else:
    display(Markdown("### QA OK\nEl brief cumple los criterios mínimos de cierre."))

print(f"QA report: {qa_path.relative_to(REPO)}")


,check,ok,detail
0,output_exists:executive_brief_1p.md,True,out/professional_pack/latest/executive_brief/e...
1,output_exists:executive_brief_1p.html,True,out/professional_pack/latest/executive_brief/e...
2,output_exists:executive_kpis.csv,True,out/professional_pack/latest/executive_brief/e...
3,output_exists:executive_caveats.csv,True,out/professional_pack/latest/executive_brief/e...
4,kpi_count_between_5_and_7,True,7 KPI rows
5,has_at_least_3_caveats,True,5 caveat rows
6,legacy_warning_if_legacy_used,True,no legacy metrics used
7,missing_metrics_are_visible,True,"Caja cierre, Deuda interna"


### QA OK
El brief cumple los criterios mínimos de cierre.

QA report: out/professional_pack/latest/executive_brief/executive_brief_qa.csv
